# GameBot: train the enemy detector on Kaggle (free GPU)
Upload your dataset zip (images/ + labels/ in YOLO format) as a Kaggle Dataset, add it to this notebook, run all cells, download the .tflite from /kaggle/working, import it in GameBot.

In [ ]:
!pip -q install ultralytics
import glob, os, shutil, zipfile, random
from pathlib import Path
print(glob.glob('/kaggle/input/*'))

In [ ]:
# 1) find the dataset (a folder with images/ and labels/, or a .zip containing them)
root = None
for d in glob.glob('/kaggle/input/*'):
    for z in glob.glob(d + '/*.zip'):
        zipfile.ZipFile(z).extractall('/kaggle/working/ds')
        root = '/kaggle/working/ds'
    if os.path.isdir(d + '/images'):
        root = d
assert root, 'dataset not found'
print('dataset root:', root)
for sub in ('', 'ds'):
    pass

In [ ]:
# 2) split train/val 90/10
imgs = sorted([p for p in Path(root).rglob('*.jpg')] + [p for p in Path(root).rglob('*.png')])
random.seed(0); random.shuffle(imgs)
n_val = max(1, len(imgs)//10)
out = Path('/kaggle/working/data')
for split, items in (('val', imgs[:n_val]), ('train', imgs[n_val:])):
    (out/'images'/split).mkdir(parents=True, exist_ok=True); (out/'labels'/split).mkdir(parents=True, exist_ok=True)
    for p in items:
        lab = Path(str(p).replace('/images/', '/labels/')).with_suffix('.txt')
        shutil.copy(p, out/'images'/split/p.name)
        if lab.exists(): shutil.copy(lab, out/'labels'/split/lab.name)
print(len(imgs), 'images')
open('/kaggle/working/data.yaml','w').write('path: /kaggle/working/data\ntrain: images/train\nval: images/val\nnames:\n  0: enemy\n  1: enemy_head\n  2: ally\n')

In [ ]:
# 3) train a nano detector
from ultralytics import YOLO
IMGSZ = 416
MODEL = 'yolo11n.pt'   # or 'yolo26n.pt' (newest; the app reads both)
m = YOLO(MODEL)
m.train(data='/kaggle/working/data.yaml', imgsz=IMGSZ, epochs=100, batch=32, project='/kaggle/working/runs', name='gamebot')

In [ ]:
# 4) export to TFLite and copy next to the notebook output
best = YOLO('/kaggle/working/runs/gamebot/weights/best.pt')
path = best.export(format='tflite', imgsz=IMGSZ, nms=False)
print(path)
shutil.copy(path, '/kaggle/working/detector.tflite')
print('download /kaggle/working/detector.tflite and import it in GameBot')